# Gradient evaluation in ParaQeet

ParaQeet builds on the idea of the *semi-automatic differentiation* [Georz2022] and combines automatic differentiaion (AD) with analytic gradients from quantum optimal control methods to provide a resource efficient and flexible framework for optimal control.
It is desgined in a top-down modular structure, where each module can be differentiated. Modules providing gradient values are derived from the base class $\texttt{Differentiable}$. All the classes deriving from $\texttt{Differentiable}$ provide a `get_value_and_gradient` method that returns the value and the gradient. 


This notebook provides an overview of the gradient computation in all the modules and aims to explain how gradients are chained across the package.

## Chain rule based gradient computation

The aim here is to compute the gradients of an objective function $(\mathcal{F})$, a $\texttt{Measurement}$ in ParaQeet, with respect to the parameters of the input pulse $(\vec{\alpha})$.
Computing this derivative for a state-transfer fidelity $(\mathcal{F}(t) = |\phi|^2 =  |\langle \lambda | \psi(t) \rangle|^2)$ for a pulse $(u(t))$ would lead to computing the terms
$$\frac{\partial \mathcal{F}(t) }{\partial \alpha} = \frac{\partial \mathcal{F}(t)}{\partial \phi}  \frac{\partial \phi}{\partial u} \frac{\partial u}{\partial \alpha}.$$

Practically the pulse contains multiple elements, and the objective function may be a sum of multiple fidelities. Computing the derivative, thus, amounts to systematic use of chain rule. 
This is computed by forward propagating the gradients through the modules
$$\frac{\partial v_k }{\partial \alpha} = \sum_{i < k} \frac{\partial v_k} {\partial v_i} \frac{\partial v_i}{\partial \alpha},$$
where $v_k$ are values of the intermediate steps, and $v_N = \mathcal{F}(t)$ for some $N$.

Thus each module computes the gradients for its own parameters, and also collects the gradients propagated through the prior modules. As shown in the flow-chart below the gradients from the device class ($\texttt{signal}$) is passed down to the $\texttt{model}$, through $\texttt{propagation}$ and finally to the $\texttt{measurement}$.

<p align="center">
  <img src="../docs/layers.svg" />
</p>

Below we explain the gradient flow across the package by using some examples.

## 1. The $\texttt{signal}$ module
Here signal generation is modelled by modeling the components used in classical signal generation, including $\texttt{Waveform}$ generator, $\texttt{LocalOscillator}$, $\texttt{IQMixer}$, etc. 

The $\texttt{Waveform}$ class derives from the $\texttt{Optimizable}$ and $\texttt{Differentiable}$ base classes and offers the `get_value` and `get_value_and_gradient` methods. This class implements a fallback to AD for computing gradients incase a `get_value_and_gradient` method is not specified. It expects the `_evaluate` method to be a pure-JAX function, with arguments as `jax.Array`. Further, all the classes deriving from $\texttt{Waveform}$ support differentiation of complex output.


The $\texttt{signal}$ module includes the $\texttt{Envelopes}$ class which defines the shape of a pulse. Lets define an example envelope of a Gaussian $$\Omega(t) = A \, e^{-\frac{(t - \mu)^2}{2\sigma^2}}.$$ And try to plot its gradients.

In [ ]:
from functools import partial

import jax.numpy as jnp
import numpy as np
from jax import jit

from paraqeet import Array, Quantity
from paraqeet.signal.envelopes import Envelope


class ExampleEnvelope(Envelope):
    _amplitude: Quantity
    _t_final: Quantity
    _t_up: Quantity
    _t_down: Quantity
    _ramp_time: Quantity

    def __init__(self):
        self._amplitude = Quantity(
            20e6,
            min_value=jnp.array(0.0),
            max_value=jnp.array(50e6),
            name="amplitude",
        )

        self._mu = Quantity(
            12e-9,
            min_value=jnp.array(0.0),
            max_value=jnp.array(20e-9),
            name="mu",
        )

        self._sigma = Quantity(
            5e-9,
            min_value=jnp.array(0.0),
            max_value=jnp.array(10e-9),
            name="sigma",
        )

    def get_parameters(self):
        return [self._amplitude, self._mu, self._sigma]

    @partial(jit, static_argnums=(0,))
    def _evaluate(self, amp: Array, mu: Array, sigma: Array, t: Array):
        return jnp.squeeze(amp * jnp.exp(-((t - mu) ** 2 / (2 * sigma**2))))

    def get_value(self, times: Array | float) -> Array:
        amp = self._amplitude.get_value()
        mu = self._mu.get_value()
        sigma = self._sigma.get_value()
        return self._evaluate(amp, mu, sigma, times)


tone = ExampleEnvelope()

Plotting the signal

In [ ]:
import matplotlib.pyplot as plt
from plotting import plot_signal

t_final = 20e-9
ts = np.linspace(0, t_final, 501)
sig, grads = tone.get_value_and_gradient(ts)

fig, ax = plt.subplots(1, figsize=(5, 3))
plot_signal(tone, ts, ax, linestyle="--")
plt.show()

Computing the gradients using the `get_value_and_gradient` method, we get 

In [ ]:
value, grads = tone.get_value_and_gradient(ts)
grads

The gradients are currently empty as the `OptimizationMap` is not defined yet. Lets define the optmap and add the `tone` to it. We need to "register" the parameters using `register_params_with_optimizables` method to inform the specified classes which parameters are being optimized.

In [ ]:
from paraqeet.optimization_map import OptimizationMap

optmap = OptimizationMap()
optmap.add(tone)
optmap.register_params_with_optimizables()
optmap

In [ ]:
value, grads = tone.get_value_and_gradient(ts)
grads.shape

Note that the shape of the gradients is (t, num_params). Here there are 3 parameters added to the `optmap`.

Lets compare the gradients with analytical formulas

In [ ]:
def der_gaussian(amp, mu, sigma, t):
    # Derivative w.r.t amp
    f = jnp.exp(-((t - mu) ** 2) / (2 * sigma**2))
    # Derivative w.r.t mu
    dmu = amp * f * (t - mu) / sigma**2
    # Derivative w.r.t sigma
    dsigma = amp * f * ((t - mu) ** 2 / sigma**3)
    return f, dmu, dsigma


params = tone.get_parameters()
damp, dmu, dsigma = der_gaussian(params[0].get_value(), params[1].get_value(), params[2].get_value(), ts)

fig, ax = plt.subplots(3, figsize=(5, 5), sharex=True)
ax[0].plot(ts / 1e-9, grads[:, 0, ...], linestyle="-", label="AD")
ax[0].plot(ts / 1e-9, damp, linestyle="--", label="Analytical")
ax[0].grid(True, linestyle=(1, (1, 5)), linewidth=1)
ax[0].legend()

ax[1].plot(ts / 1e-9, grads[:, 1, ...], linestyle="-", label="AD")
ax[1].plot(ts / 1e-9, dmu, linestyle="--", label="Analytical")
ax[1].grid(True, linestyle=(1, (1, 5)), linewidth=1)
ax[1].legend()

ax[2].plot(ts / 1e-9, grads[:, 2, ...], linestyle="-", label="AD")
ax[2].plot(ts / 1e-9, dsigma, linestyle="--", label="Analytical")
ax[2].grid(True, linestyle=(1, (1, 5)), linewidth=1)
ax[2].legend()

ax[0].set_ylabel(r"$\frac{\partial \Omega(t)}{\partial A}$")
ax[1].set_ylabel(r"$\frac{\partial \Omega(t)}{\partial \mu}$")
ax[2].set_ylabel(r"$\frac{\partial \Omega(t)}{\partial \sigma}$")
ax[-1].set_xlabel("Time [ns]")
plt.show()

Further, the signal module includes filter functions and $\texttt{DRAGMixer}$ to modify the pulse shape. Lets include a $\texttt{FlatTopGaussianFilter}$ $(\epsilon(t) = s(t) \Omega(t))$ on top of this pulse and add the DRAG component.
The final pulse would be $$ \mathcal{E}(t) = s(t) \Omega(t) - i \frac{\dot{\epsilon(t)}}{\delta}.$$

The derivative of this pulse wrt parameters like $\mu$ can be evaluated by
$$ \frac{\partial \mathcal{E}(t)}{\partial \mu} = s(t) \frac{\partial \Omega(t)}{\partial \mu} - \frac{i}{\delta} \frac{\partial^2  (s(t) \Omega(t))}{\partial \mu \, \partial t} $$



In [ ]:
from paraqeet.signal.waveform import DRAGMixer, FlatTopGaussianFilter

drag_tone = DRAGMixer(tone)
filtered_tone = FlatTopGaussianFilter(
    envelopes=drag_tone, t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final")
)

optmap.remove(tone)
optmap.add(drag_tone)
optmap.register_params_with_optimizables()
optmap

In [ ]:
fig, ax = plt.subplots(1, figsize=(5, 3))

plot_signal(drag_tone, ts, ax, linestyle="--")
plt.show()

Lets plot the derivative with $\mu$

In [ ]:
value, grads = drag_tone.get_value_and_gradient(ts)

fig, ax = plt.subplots(1, figsize=(5, 3), sharex=True)
ax.plot(ts / 1e-9, np.real(grads[:, 1, ...]), linestyle="--", label="Real")
ax.plot(ts / 1e-9, np.imag(grads[:, 1, ...]), linestyle="--", label="Imag")
ax.grid(True, linestyle=(1, (1, 5)), linewidth=1)
ax.legend()
ax.set_xlabel("Time [ns]")
plt.show()

Now there is the added imaginary component in the derivative.

Finally, the `Generator` class gathers the gradients from individual components and returns them.

In [ ]:
from paraqeet.signal.iq_mixer import IQMixer

gen = IQMixer(envelopes=[tone])

optmap.remove(drag_tone)
optmap.add(gen)
optmap.register_params_with_optimizables()
optmap

In [ ]:
value, grads = gen.get_value_and_gradient(ts)
grads.shape

Note that now that the optmap has 5 elements, the gradients generated by the generator is also 5.

## 2. The $\texttt{Model}$ module

$\texttt{Model}$ includes the Hamiltonian, Drive and coupling. Due to the parameter dependence are only in the coefficient of the operators, computing derivatives just results in multiplying the signal gradient with the respective operator. For a Hamiltonian of the from $H(t) = H_0 + u(\alpha, t) H_d$, where $u(\alpha, t)$ is the drive signal
$$ \frac{\partial H(t)}{\partial \alpha} = \frac{\partial u(t)}{\partial \alpha} H_d.$$

Note that similar operation can also be performed to compute the gradients with respect to the model parameters, for e.g., the the coupling strength between two subsystems, by adding the corresponding quantity to the optmap. Thus we can put the optimization of the model and pulse parameter on the same footing.

For this example, lets define a qubit, given by the Hamiltonian
$$H(t)=H_\text{drift}+H_c(t)= \frac{\omega_q}{2} \sigma_z + \Omega(t)\sigma_x, $$
and add its frequency and drive parameters to the optmap.

In [ ]:
from paraqeet.model.drive import DriveOperator
from paraqeet.model.qubit import Qubit

freq_q = 4.8e9
omega_q = 2 * np.pi * freq_q

drive = DriveOperator(gen, is_longitudinal=False)
controlled_qubit = Qubit(
    frequency=Quantity(omega_q, 0.8 * omega_q, 1.2 * omega_q, unit="Hz", name="Qubit Freq"), drives=[drive]
)

model_params = controlled_qubit.get_parameters()

In [ ]:
optmap.add(controlled_qubit, model_params[-1])
optmap.register_params_with_optimizables()
optmap

We see now that the optmap has an additional parameter. Let's compute the gradient of the Hamiltonian and check the number of components.

In [ ]:
value, grads = controlled_qubit.get_hamiltonian_and_gradient(ts)
grads.shape

The gradients now has the shape (t, num_params, dim, dim), where dim is the dimension of the Hilbert space. We see that the gradients now has 6 parameters as expected.

Similarly the equation of motion (EOM) returns the gradients of the right hand side of the EOM.

In [ ]:
from paraqeet.model.schroedinger_equation import SchroedingerEquation

schrgl = SchroedingerEquation(
    hamiltonian_func=controlled_qubit.get_hamiltonian,
    hamiltonian_and_gradient_func=controlled_qubit.get_hamiltonian_and_gradient,
)

value, grads = schrgl.get_value_and_gradient(ts)
grads.shape

## 3. The $\texttt{Propagation}$ Module

The $\texttt{Propagation}$ module computes the derivative of the unitary operator generated by the time-dependent Hamiltonian by using quantum optimal control methods like GRAPE and GOAT. These methods work for specific pulse ansatz, for e.g. GRAPE require a piecewise constant (PWC) ansatz, and GOAT requires a continuous ansatz. These methods provide ways to propagate the derivative of the Hamiltonian to compute derivative of the state-overlap and the unitary operator, respectively.

The gradients computed in GRAPE are computed by:
$$\frac{\partial \Phi(T)}{\partial u_n} = \bra{\tilde{\lambda}(t_n)} \frac{\partial U_n}{\partial u_n} \ket{\psi(t_n)} $$
where $u_n = u(t_n)$ is the PWC pulse value, $\ket{\tilde{\lambda}(t)}$ the reverse-propagated target state, $\ket{\psi(t)}$ the forward propagated initial state, and $\Phi(T) = \langle{\lambda}|{\psi(T)}\rangle$ the state-overlap.

The gradients computed in GOAT are computed by the solution of:
$$\frac{\partial}{\partial t}
\begin{pmatrix}
U(t) \\[4pt]
\partial_{\alpha_k} U(t)
\end{pmatrix}
= -i
\begin{pmatrix}
H(\vec{\alpha}, t) & 0 \\[4pt]
\partial_{\alpha_k} H(\vec{\alpha}, t) & H(\vec{\alpha}, t)
\end{pmatrix}
\begin{pmatrix}
U(t) \\[4pt]
\partial_{\alpha_k} U(t)
\end{pmatrix}$$


In this example we look at the gradients computed using GOAT by using the propagation module $\texttt{ScipyExpmGOAT}$.